# 09 — Final Test Set Evaluation

**THIS NOTEBOOK RUNS ONCE. RESULTS ARE LOCKED AFTER EXECUTION.**

Evaluates the best M8 checkpoint (selected by highest AP_hard on the val set) on the held-out KITTI test split.

**Protocol:**
1. Check that `results/.test_run_complete` does NOT exist (prevents re-runs)
2. Load the best M8 checkpoint (highest AP_hard on validation)
3. Run inference on the KITTI test split with no gradients
4. Compute: AP_easy, AP_mod, AP_hard, ORS, FN_rate_hard, mAP@0.5, mAP@.5:.95, FPS
5. Save to `results/tables/test_results_FINAL.csv`
6. Create `results/.test_run_complete` flag

**Prerequisites:**
- Notebook `07_full_system` must have completed (M8 checkpoint exists)
- Notebook `08_ablation` should have completed (final model selection confirmed)
- `results/.test_run_complete` must NOT exist

⚠️ **Do not re-run this notebook after it has completed successfully.**

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.config import load_config, set_all_seeds
from src.datasets import KITTIDataset, collate_fn
from src.metrics import (
    compute_kitti_ap, compute_ors, compute_fn_rate_hard,
    compute_fps, sample_to_annotation
)

ROOT = Path('..').resolve()
RESULTS_DIR = ROOT / 'results'
TABLES_DIR = RESULTS_DIR / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)

FLAG_PATH = RESULTS_DIR / '.test_run_complete'

# ── Guard: refuse to run if already complete ──────────────────────────────────
if FLAG_PATH.exists():
    raise RuntimeError(
        f'Test set has already been evaluated.\n'
        f'Flag file: {FLAG_PATH}\n'
        f'DO NOT RE-RUN. Results are in: {TABLES_DIR / "test_results_FINAL.csv"}'
    )

print('Pre-run check passed: test set not yet evaluated.')
print(f'Flag will be created at: {FLAG_PATH}')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
cfg = load_config(ROOT / 'configs' / 'full_system.yaml', overrides={
    'run_mode': 'eval',
})
set_all_seeds(cfg.seed)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Find best M8 checkpoint (highest AP_hard on val)
best_ckpt = cfg.checkpoint_dir / 'M8' / 'weights' / 'best.pt'
if not best_ckpt.exists():
    raise FileNotFoundError(
        f'M8 best checkpoint not found: {best_ckpt}\n'
        'Run notebook 07_full_system first.'
    )
print(f'Loading checkpoint: {best_ckpt}')

In [ ]:
# ── Load model and test dataset ───────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO(str(best_ckpt))
model.model.eval()

test_ds = KITTIDataset(cfg.kitti_root, split='test', imgsz=cfg.imgsz)
test_loader = DataLoader(
    test_ds, batch_size=cfg.batch, shuffle=False,
    collate_fn=collate_fn, num_workers=4,
    pin_memory=(device == 'cuda')
)

print(f'Test set size: {len(test_ds)} images')

In [ ]:
# ── Inference on test set (NO GRADIENTS) ──────────────────────────────────────
from tqdm.auto import tqdm

predictions, annotations = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test inference'):
        imgs = batch['image'].to(device)
        results = model(imgs, verbose=False)
        for i, r in enumerate(results):
            boxes  = r.boxes.xyxyn.cpu().numpy() if r.boxes is not None and len(r.boxes) > 0 else []
            scores = r.boxes.conf.cpu().numpy()  if r.boxes is not None and len(r.boxes) > 0 else []
            predictions.append({
                'image_id': batch['image_id'][i],
                'boxes': boxes,
                'scores': scores,
            })
            annotations.append(sample_to_annotation(batch, i))

print(f'Collected {len(predictions)} prediction sets.')

In [ ]:
# ── Compute all metrics ───────────────────────────────────────────────────────
print('Computing metrics...')

ap_easy  = compute_kitti_ap(predictions, annotations, 'easy')
ap_mod   = compute_kitti_ap(predictions, annotations, 'moderate')
ap_hard  = compute_kitti_ap(predictions, annotations, 'hard')
ors      = compute_ors(predictions, annotations)
fn_hard  = compute_fn_rate_hard(predictions, annotations)
fps_info = compute_fps(model.model, input_size=(cfg.imgsz, cfg.imgsz), device=device)

# mAP@0.5 and mAP@.5:.95 from Ultralytics val (on test split)
val_metrics = model.val(
    data=str(ROOT / 'data' / 'kitti_ultralytics.yaml'),
    split='test', imgsz=cfg.imgsz, device=device, verbose=False
)
map50    = float(val_metrics.box.map50)    * 100
map5095  = float(val_metrics.box.map)      * 100

results_row = {
    'model_id':      'M8',
    'AP_easy':       ap_easy,
    'AP_mod':        ap_mod,
    'AP_hard':       ap_hard,
    'ORS':           ors,
    'FN_rate_hard':  fn_hard,
    'mAP50':         map50,
    'mAP5095':       map5095,
    'FPS_mean':      fps_info['mean_fps'],
    'FPS_mean_ms':   fps_info['mean_ms'],
    'FPS_std':       fps_info['std_fps'],
}

print('\n' + '='*60)
print('FINAL TEST SET RESULTS — DO NOT RE-RUN')
print('='*60)
for k, v in results_row.items():
    print(f'  {k:<20}: {f"{v:.4f}" if isinstance(v, float) else v}')

In [ ]:
# ── Save results and lock ─────────────────────────────────────────────────────
out_csv = TABLES_DIR / 'test_results_FINAL.csv'
pd.DataFrame([results_row]).to_csv(out_csv, index=False)
print(f'Results saved to: {out_csv}')

# Create lock flag — prevents re-evaluation
FLAG_PATH.touch()
print(f'Lock file created: {FLAG_PATH}')
print('\nTest evaluation complete and locked. Proceed to notebook 10_paper_figures.')